# Spam I: Discrete Naive Bayes by hand

**Goal.** Build the smallest possible spam filter, with nothing but Python, and understand every number it produces.
This is the first rung of a ladder that ends with small-language-model embeddings (see the next notebooks).

**Attacker's view.** A filter that is simple enough to compute by hand is also simple enough to *reverse engineer*:
once you know that every word contributes an independent vote, you know exactly which words to add to get a message through.
We will exploit that in notebook 07.

Probability refresher: `../00-intro/notebook_00.ipynb`.

## 1. Bayes' theorem for text

A message is a list of words $W_1,\dots,W_n$; its class is $y\in\{\text{spam},\text{ham}\}$. Bayes' theorem gives the
probability of *spam given the words*:

$$
P(y \mid W_1,\dots,W_n)=\frac{P(W_1,\dots,W_n \mid y)\,P(y)}{P(W_1,\dots,W_n)}
$$

Estimating the joint likelihood $P(W_1,\dots,W_n\mid y)$ is hopeless (almost no message repeats). The **naive** assumption:
words are *conditionally independent given the class*,

$$
P(W_1,\dots,W_n \mid y)=\prod_{i=1}^{n}P(W_i\mid y)
$$

and the evidence $P(W_1,\dots,W_n)$ is obtained by summing over the two classes (law of total probability):

$$
P(\text{spam}\mid W)=\frac{P(\text{spam})\prod_i P(W_i\mid\text{spam})}
{P(\text{spam})\prod_i P(W_i\mid\text{spam})+P(\text{ham})\prod_i P(W_i\mid\text{ham})}
$$

In this notebook a word *counts* once per message (**discrete / presence** features):
$P(w\mid y)=\dfrac{\#\{\text{messages of class } y \text{ containing } w\}+k}{\#\{\text{messages of class } y\}+2k}$
where $k=1$ is the **Laplace smoothing** constant (it prevents a zero probability for unseen words).

In [ ]:
from functools import reduce

train = [
    ("send us your password", "spam"),
    ("review our website", "spam"),
    ("send your password", "spam"),
    ("send us your account", "spam"),
    ("your activity report", "ham"),
    ("benefits physical activity", "ham"),
    ("the importance vows", "ham"),
]
test = [
    ("renew your password", "spam"),
    ("renew your vows", "spam"),
    ("benefits of our account", "ham"),
    ("the importance of physical activity", "ham"),
]

## 2. Vocabulary and counts

In [ ]:
def tokenize(text):
    return text.lower().split()


def vocabulary(messages):
    return sorted({w for m in messages for w in tokenize(m)})


spam_msgs = [m for m, y in train if y == "spam"]
ham_msgs = [m for m, y in train if y == "ham"]

print("spam vocabulary:", vocabulary(spam_msgs))
print("ham vocabulary :", vocabulary(ham_msgs))

## 3. Likelihoods $P(w\mid y)$ with Laplace smoothing

In [ ]:
def message_frequency(messages):
    """Number of messages that contain each word (a word counts once per message)."""
    counts = {}
    for m in messages:
        for w in set(tokenize(m)):
            counts[w] = counts.get(w, 0) + 1
    return counts


K = 1  # Laplace smoothing
spam_df = message_frequency(spam_msgs)
ham_df = message_frequency(ham_msgs)


def p_word(word, counts, n_messages, k=K):
    return (counts.get(word, 0) + k) / (n_messages + 2 * k)


for w in ["password", "your", "activity", "renew"]:
    print(f"{w:9} P(w|spam)={p_word(w, spam_df, len(spam_msgs)):.3f}   P(w|ham)={p_word(w, ham_df, len(ham_msgs)):.3f}")

`renew` never appeared in training: smoothing gives it a small but non-zero probability in *both* classes, so an unseen
word cannot veto the whole message.

## 4. Priors $P(y)$

In [ ]:
prior_spam = len(spam_msgs) / len(train)
prior_ham = len(ham_msgs) / len(train)
print(f"P(spam)={prior_spam:.3f}  P(ham)={prior_ham:.3f}")

## 5. Posterior: prior $\times$ likelihoods, normalised

Only the words *present* in the message contribute (a presence-only model).

In [ ]:
def posterior_spam(text):
    words = tokenize(text)
    like_spam = reduce(lambda a, b: a * b, [p_word(w, spam_df, len(spam_msgs)) for w in words], 1.0)
    like_ham = reduce(lambda a, b: a * b, [p_word(w, ham_df, len(ham_msgs)) for w in words], 1.0)
    num = prior_spam * like_spam
    return num / (num + prior_ham * like_ham)


def predict(text):
    p = posterior_spam(text)
    return ("spam" if p >= 0.5 else "ham"), p

## 6. Classify the toy data

In [ ]:
def evaluate(dataset, verbose=True):
    hits = 0
    for text, label in dataset:
        pred, p = predict(text)
        hits += pred == label
        if verbose:
            print(f"{'ok ' if pred == label else 'ERR'} {text:40} P(spam)={p:.3f}  -> {pred}")
    return hits / len(dataset)


print("train accuracy:", evaluate(train))
print()
print("test accuracy :", evaluate(test))

The model gets the two easy messages right and misses two. *"renew your vows"* is labelled spam only because of the
words it shares with the spam class, and the model has no notion of meaning (`vows` is ham-only, so it wins). *"benefits of
our account"* is flagged as spam because `our` and `account` occur only in spam. **Every word is an independent vote**, and
that is precisely what an attacker exploits.

## 7. Exercises

1. Change `K` to `0.1` and to `10`. How do the posteriors of the four test messages change? Why does a large `K` make the
   model *less sure*?
2. Add the ham message `"renew your account benefits"` to the training data. Which test posteriors move most?
3. **Attacker.** Append one word to `"renew your password"` to make it *ham*. Which word did you pick and why (look at the
   ratio $P(w\mid\text{ham})/P(w\mid\text{spam})$)?
4. Compute by hand $P(\text{spam}\mid\text{"send your password"})$ and compare with the code.